# Causal Multi-Head Self-Attention

Multi-head attention extends self-attention by projecting query, key, and value vectors into multiple independent representation subspaces. This notebook implements an efficient multi-head causal attention module that computes all attention heads in parallel using tensor reshaping and combines their outputs via a final linear projection layer.

## Table of Contents
1. Imports
2. Causal Multi-Head Attention Implementation
3. Verification and Multi-Head Shape Testing


## 1. Imports

Imports PyTorch modules for tensor manipulation, linear transformations, and dropout regularization.


In [7]:
import torch 
import math 
import torch.nn as nn 
import torch.nn.functional as F

## 2. Causal Multi-Head Attention Implementation

Implements `CausalMultiHeadSelfAttention`. For model dimension $d_{model}$ and head count $h$, each head operates on subspace dimension $d_{head} = d_{model} / h$:

1. Projections $Q, K, V$ of shape `(B, T, d_model)` are reshaped and transposed to `(B, h, T, d_head)`.
2. Batched scaled dot-product attention with causal masking is computed in parallel across all heads.
3. Head outputs are transposed and concatenated back to `(B, T, d_model)`.
4. A final linear projection $W_o \in \mathbb{R}^{d_{model} \times d_{model}}$ combines the multi-head representations.


In [8]:
class CausalMultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, num_heads, context_length=1024, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_query = nn.Linear(d_model, d_model, bias=qkv_bias)
        self.W_key = nn.Linear(d_model, d_model, bias=qkv_bias)
        self.W_value = nn.Linear(d_model, d_model, bias=qkv_bias)
        self.out_proj = nn.Linear(d_model, d_model, bias=True)

        self.dropout = nn.Dropout(dropout)

        # Register persistent causal mask buffer
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x, return_attention_weights=False):
        batch_size, num_tokens, _ = x.shape

        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        Q = Q.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)

        # Causal mask up to sequence length
        causal_mask = self.mask[:num_tokens, :num_tokens]
        scores = scores.masked_fill(causal_mask, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = attention_weights @ V
        context = context.transpose(1, 2).contiguous().view(
            batch_size, num_tokens, self.d_model
        )
        
        output = self.out_proj(context)

        if return_attention_weights:
            return output, attention_weights
        return output


## 3. Verification and Multi-Head Shape Testing

Verifies the module with batch size 2, sequence length 8, model dimension 256, and 8 attention heads. Verifies that input tensor shape `(2, 8, 256)` matches the output tensor shape `(2, 8, 256)`.


In [9]:
batch_size = 2
num_tokens = 8
d_model = 256
num_heads = 8

x = torch.randn(batch_size, num_tokens, d_model)

attention = CausalMultiHeadSelfAttention(
    d_model=d_model,
    num_heads=num_heads
)

output, attention_weights = attention(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention shape:", attention_weights.shape)

Input shape: torch.Size([2, 8, 256])
Output shape: torch.Size([2, 8, 256])
Attention shape: torch.Size([2, 8, 8, 8])
